# 01 — Syndrome Exploration

Visualize detector event matrices from QSSF recordings: heatmaps, weight distributions,
and per-ancilla statistics.

**Requirements:** `pip install stabstream numpy matplotlib`  
**Optional:** A `.qssf` recording file. Synthetic data is generated automatically if none is provided.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Change this to the path of your recording, or leave as None to use synthetic data.
RECORDING_PATH = None  # e.g. "../data/surface_d5.qssf"
N_ROUNDS = 200
N_ANCILLAS = 24  # d=5 surface code
P_PHYSICAL = 0.005

## 1. Load detector events

In [ ]:
def load_from_file(path: str) -> np.ndarray:
    """Load detector events from a QSSF file → (rounds, ancillas) bool array."""
    from stabstream.io import load_qssf_batch
    batches = list(load_qssf_batch(path, batch_size=1000))
    if not batches:
        raise ValueError(f"No frames found in {path}")
    return np.vstack(batches)


def synthetic_syndrome_matrix(
    rounds: int, ancillas: int, p: float, seed: int = 42
) -> np.ndarray:
    """Generate a synthetic (rounds, ancillas) syndrome matrix at error rate ~2p."""
    rng = np.random.default_rng(seed)
    # Each ancilla fires with probability ~2p (one error can flip two ancillas)
    return rng.random((rounds, ancillas)) < 2 * p


if RECORDING_PATH and Path(RECORDING_PATH).exists():
    matrix = load_from_file(RECORDING_PATH)
    N_ROUNDS, N_ANCILLAS = matrix.shape
    print(f"Loaded {N_ROUNDS} rounds × {N_ANCILLAS} ancillas from {RECORDING_PATH}")
else:
    matrix = synthetic_syndrome_matrix(N_ROUNDS, N_ANCILLAS, P_PHYSICAL)
    print(f"Using synthetic data: {N_ROUNDS} rounds × {N_ANCILLAS} ancillas, p={P_PHYSICAL}")

print(f"Global fire frequency: {matrix.mean():.4f}")

## 2. Detector events heatmap

In [ ]:
from stabstream.plot import plot_syndrome_heatmap

fig, ax = plt.subplots(figsize=(12, 5))
plot_syndrome_heatmap(
    matrix[:100],           # show first 100 rounds
    ax=ax,
    title=f"Detector Events (first 100 rounds, p={P_PHYSICAL})",
    cmap="Blues",
)
plt.tight_layout()
plt.savefig("syndrome_heatmap.png", dpi=150)
plt.show()

## 3. Syndrome weight distribution

In [ ]:
from stabstream.plot import plot_syndrome_weight_hist

weights = matrix.sum(axis=1).astype(int)  # fired detectors per round

fig, ax = plt.subplots(figsize=(8, 4))
plot_syndrome_weight_hist(weights, ax=ax, title="Syndrome Weight Distribution")

mean_w = weights.mean()
expected_w = 2 * P_PHYSICAL * N_ANCILLAS  # rough Poisson mean
ax.axvline(mean_w, color="red", linestyle="--", linewidth=1.5, label=f"Mean = {mean_w:.2f}")
ax.axvline(expected_w, color="green", linestyle=":", linewidth=1.5,
           label=f"Expected ~2pn = {expected_w:.2f}")
ax.legend()

plt.tight_layout()
plt.savefig("syndrome_weight_hist.png", dpi=150)
plt.show()

print(f"Weight statistics: mean={mean_w:.2f}, std={weights.std():.2f}, max={weights.max()}")

## 4. Per-ancilla fire frequency

In [ ]:
from stabstream.plot import plot_fire_frequency

freqs = matrix.mean(axis=0)  # shape (ancillas,)

fig, ax = plt.subplots(figsize=(12, 4))
plot_fire_frequency(
    freqs,
    ax=ax,
    title="Per-Ancilla Fire Frequency",
    expected_rate=2 * P_PHYSICAL,
    outlier_threshold=3.0,
)
plt.tight_layout()
plt.savefig("fire_frequency.png", dpi=150)
plt.show()

mean_f = freqs.mean()
std_f = freqs.std()
outliers = np.where(np.abs(freqs - mean_f) > 3 * std_f)[0]
if len(outliers):
    print(f"Outlier ancillas (>3σ): {outliers.tolist()}")
    for i in outliers:
        print(f"  A{i}: freq={freqs[i]:.4f}  ({(freqs[i]-mean_f)/std_f:+.1f}σ)")
else:
    print("No outlier ancillas detected (all within 3σ).")

## 5. 2-D layout heatmap (optional)

If you have a `HardwareSchema` with qubit layout coordinates, you can visualize
per-ancilla frequencies on the code geometry.

In [ ]:
# Build a 5×5 grid layout for a d=5 surface code (24 ancillas on a √24 ≈ 5×5 grid)
# In practice, load layout from dem.to_schema_json() or a HardwareSchema JSON file.
rows, cols = 4, 6  # 24 ancillas on a 4×6 grid
xs, ys = np.meshgrid(range(cols), range(rows))
layout = np.stack([xs.ravel(), ys.ravel()], axis=1)  # shape (24, 2)

fig, ax = plt.subplots(figsize=(7, 5))
plot_fire_frequency(
    freqs,
    layout=layout,
    ax=ax,
    title="Per-Ancilla Fire Frequency (2-D layout)",
    expected_rate=2 * P_PHYSICAL,
)
plt.tight_layout()
plt.savefig("fire_frequency_2d.png", dpi=150)
plt.show()

## 6. Temporal autocorrelation

Check whether errors cluster in time (indicating leakage or slow drift).

In [ ]:
max_lag = 20
global_signal = matrix.mean(axis=1)  # mean fire rate per round
autocorr = np.correlate(
    global_signal - global_signal.mean(),
    global_signal - global_signal.mean(),
    mode='full'
)
autocorr = autocorr[autocorr.size // 2:]  # keep positive lags
autocorr /= autocorr[0]  # normalize

fig, ax = plt.subplots(figsize=(8, 3))
ax.stem(range(max_lag + 1), autocorr[:max_lag + 1], linefmt="C0-",
        markerfmt="C0o", basefmt="k-")
ax.axhline(2 / np.sqrt(N_ROUNDS), color="red", linestyle="--",
           linewidth=1, alpha=0.7, label="95% CI")
ax.axhline(-2 / np.sqrt(N_ROUNDS), color="red", linestyle="--",
           linewidth=1, alpha=0.7)
ax.set_xlabel("Lag (rounds)")
ax.set_ylabel("Autocorrelation")
ax.set_title("Global Fire Rate Autocorrelation")
ax.legend()
plt.tight_layout()
plt.savefig("autocorrelation.png", dpi=150)
plt.show()

if autocorr[1] > 2 / np.sqrt(N_ROUNDS):
    print("⚠  Lag-1 autocorrelation is significant — possible temporal clustering (leakage?).")
else:
    print("Lag-1 autocorrelation within noise — no significant temporal clustering.")